# Chapter 4: Probability & Statistics for Machine Learning
## Notebook 01 - Introduction

Probability is the mathematical foundation of uncertainty—and ML systems deal with uncertainty everywhere: predictions, model selection, A/B testing, and confidence in results.

**What you'll learn:**
- What is probability? Sample spaces, events
- Conditional probability with AI relevance: P(spam|word)
- Independence and the Law of Large Numbers
- Interactive experiments: predict before running!

**Time estimate:** 2.5 hours

---
*Generated by Berta AI | Created by Luigi Pascal Rondanini*

## 1. What is Probability?

From coins and dice we build intuition, then apply to AI:
- **Coins**: P(heads) = 0.5 — binary outcomes
- **Dice**: P(rolling 6) = 1/6 — discrete outcomes
- **AI**: P(model correct), P(spam|"viagra"), P(conversion|variant A)

**Predict**: What happens when we flip a fair coin 10 times? How many heads do you expect?

In [ ]:
import random

def simulate_coin_flips(n_flips):
    """Simulate n fair coin flips. 1=heads, 0=tails."""
    return [random.randint(0, 1) for _ in range(n_flips)]

# Run experiment
flips = simulate_coin_flips(10)
heads = sum(flips)
print(f"10 flips: {flips}")
print(f"Heads: {heads}/10 = {heads/10:.2f}")
print("Expected: ~5 (0.5 × 10)")

## 2. Sample Spaces and Events

```mermaid
flowchart TD
    A[Sample Space Ω] --> B[Event A]
    A --> C[Event B]
    B --> D[P(A) = |A|/|Ω|]
    C --> E[P(B) = |B|/|Ω|]
    B --> F[A ∩ B]
    C --> F
    F --> G[P(A and B)]
```

For a 6-sided die: Ω = {1,2,3,4,5,6}. Event "even" = {2,4,6} → P(even) = 3/6 = 0.5.

In [ ]:
def simulate_dice(n_rolls):
    """Simulate n dice rolls. Returns list of outcomes 1-6."""
    return [random.randint(1, 6) for _ in range(n_rolls)]

rolls = simulate_dice(60)
even_count = sum(1 for r in rolls if r % 2 == 0)
print(f"60 rolls: evens = {even_count}, P(even) ≈ {even_count/60:.3f}")
print("Expected: 0.5")

## 3. Conditional Probability: P(A|B)

**P(A|B) = P(A and B) / P(B)** — probability of A given B occurred.

**AI Example**: Spam classification. P(spam | "viagra" in email)?
- If "viagra" appears, how likely is it spam?
- This is the core of Naive Bayes classifiers!

**Predict**: If P(spam)=0.3, P("viagra"|spam)=0.8, P("viagra"|ham)=0.01, what is P(spam|"viagra")?

In [ ]:
def conditional_probability(p_a_and_b, p_b):
    """P(A|B) = P(A∩B) / P(B)"""
    if p_b == 0:
        return 0.0
    return p_a_and_b / p_b

# Spam example: P(spam)=0.3, P(viagra|spam)=0.8, P(viagra|ham)=0.01
p_spam = 0.3
p_viagra_given_spam = 0.8
p_viagra_given_ham = 0.01
p_ham = 1 - p_spam

# P(viagra) = P(viagra|spam)*P(spam) + P(viagra|ham)*P(ham)
p_viagra = p_viagra_given_spam * p_spam + p_viagra_given_ham * p_ham
# P(spam and viagra) = P(viagra|spam) * P(spam)
p_spam_and_viagra = p_viagra_given_spam * p_spam

p_spam_given_viagra = conditional_probability(p_spam_and_viagra, p_viagra)
print(f"P(spam | 'viagra') = {p_spam_given_viagra:.3f}")
print("→ Very high! Word 'viagra' strongly indicates spam.")

## 4. Probability Concepts Hierarchy

```mermaid
flowchart TB
    subgraph Foundational
        A[Sample Space Ω]
        B[Events]
        C[Probability Axioms]
    end
    subgraph Derived
        D[Conditional P(A|B)]
        E[Independence P(A∩B)=P(A)P(B)]
        F[Bayes Theorem]
    end
    A --> D
    B --> D
    D --> E
    D --> F
```

## 5. Independence

Events A and B are **independent** if P(A and B) = P(A) × P(B).

Then P(A|B) = P(A) — knowing B doesn't change our belief about A.

**AI relevance**: Naive Bayes assumes feature independence (often violated but works well).

In [ ]:
import matplotlib.pyplot as plt

# Bar chart: probability of each die face
faces = [1, 2, 3, 4, 5, 6]
counts = [rolls.count(f) for f in faces]
probs = [c/60 for c in counts]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(faces, probs, color='steelblue', edgecolor='navy')
ax.axhline(y=1/6, color='red', linestyle='--', label='Theoretical 1/6')
ax.set_xlabel('Die Face')
ax.set_ylabel('Empirical Probability')
ax.set_title('Distribution of 60 Dice Rolls')
ax.legend()
plt.tight_layout()
plt.savefig('../assets/diagrams/dice_distribution.svg')
plt.show()
print("Plot saved to assets/diagrams/dice_distribution.svg")

## 6. Law of Large Numbers

**Predict**: As we increase the number of coin flips, what happens to the proportion of heads?

The LLN says: sample mean → true mean as n → ∞. Let's simulate and visualize!

In [ ]:
import numpy as np

def running_proportion(flips):
    """Cumulative proportion of heads at each step."""
    return np.cumsum(flips) / np.arange(1, len(flips)+1, dtype=float)

np.random.seed(42)
n = 1000
flips = np.random.randint(0, 2, n)
proportions = running_proportion(flips)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, n+1), proportions, alpha=0.8, color='steelblue')
ax.axhline(y=0.5, color='red', linestyle='--', label='True P(heads)=0.5')
ax.set_xlabel('Number of flips')
ax.set_ylabel('Proportion of heads')
ax.set_title('Law of Large Numbers: Convergence of Sample Proportion')
ax.legend()
ax.set_xlim(1, n)
plt.tight_layout()
plt.savefig('../assets/diagrams/lln_convergence.svg')
plt.show()
print("Sample proportion converges toward 0.5 as n increases.")

## 7. Probability Heatmap: Joint Distribution

Visualize P(A, B) for two events. **Predict**: For two independent coins, what does the joint distribution look like?

In [ ]:
# Simulate pairs: (coin1, coin2). Outcomes: (0,0), (0,1), (1,0), (1,1)
pairs = [(random.randint(0,1), random.randint(0,1)) for _ in range(1000)]
from collections import Counter
counts = Counter(pairs)

heatmap = np.zeros((2, 2))
for (a, b), c in counts.items():
    heatmap[a, b] = c / 1000

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(heatmap, cmap='Blues', vmin=0, vmax=0.35)
ax.set_xticks([0, 1]); ax.set_xticklabels(['Tails', 'Heads'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['Tails', 'Heads'])
ax.set_xlabel('Coin 2')
ax.set_ylabel('Coin 1')
ax.set_title('Joint P(Coin1, Coin2) — Independent Coins')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{heatmap[i,j]:.2f}', ha='center', va='center', color='white', fontsize=14)
plt.colorbar(im, label='Probability')
plt.tight_layout()
plt.savefig('../assets/diagrams/prob_heatmap.svg')
plt.show()
print("Each cell ≈ 0.25 for independent fair coins.")

## 8. Summary

You've seen:
- **Sample spaces and events** — coins, dice, then AI examples
- **Conditional probability** P(A|B) — spam classification
- **Independence** — when P(A|B) = P(A)
- **Law of Large Numbers** — empirical proportions converge to true probabilities

Next: probability distributions (Bernoulli, Binomial, Normal, Poisson) and Bayes' theorem.

---
*Generated by Berta AI | Created by Luigi Pascal Rondanini*